### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
# Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """process all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("***/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error : {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

In [ ]:
all_pdf_documents = process_all_pdfs("../data")

In [ ]:
all_pdf_documents

In [ ]:
### Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks = split_documents(all_pdf_documents)
# chunks

### Embedding and VectorStoreDB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Arg:
            model_name: HuggingFace model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loadding embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

### VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        """
        Initialize the vector

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embedding for RAG"},
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error Initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # document content
            documents_text.append(doc.page_content)

            # embedding
            embeddings_list.append(embedding.tolist())

        # add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store : {e}")
            raise


vectorStore = VectorStore()
vectorStore

In [ ]:
## convert the text to embeddings
texts=[doc.page_content for doc in chunks]
# texts

## generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)
# embeddings

## store init vector database
vectorStore.add_documents(chunks, embeddings)

### Retriver Pipeline From VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrievak frin tge vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self, query: str, top_k: int = 5, score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query (str): The search query
            top_k (int, optional): Number of top results to return. Defaults to 5.
            score_threshold (float, optional): Minimun similarity score threshold. Defaults to 0.0.

        Returns:
            List[Dict[str,Any]]: List of dictionaries containing retrieved documents and metadata
        """

        # generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], n_results=top_k
            )

            # process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # convert distance to similarity score(ChromDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                id: doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1,
                            }
                        )
                print(f"Retrieved {len(retrieved_docs)} documents(after filtering)")
            else:
                print("No documents found")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(vectorStore, embedding_manager)

In [ ]:
rag_retriever.retrieve("What is attention is all you need")

In [ ]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")


### Integration Vectordb Context pipeline with LLM output

In [ ]:
### Simple RAG pipeline with Grop LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

# Initialize the grop LLM (set your GROQ_API_KEY in environment)
grop_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    api_key=grop_api_key,
    model_name="qwen/qwen3.8-27b",
    temperature=0.1,
    max_tokens=1024,
)


### 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    # retrieve the context
    results = retriever.retrieve(query, top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        return "No relevant context to answer the question concisely"
    
    # generate the answer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.
    Context:
    {context}
    
    Question: {query}
    Answer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    return response

In [ ]:
answer= rag_simple("What is attention mechanism?", rag_retriever,llm)
answer

### Enhanced RAG Pipeline Features

In [ ]:
def rag_advance(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.

    Args:
        query (str): query for search
        retriever (RAGRetriever): instance of reg retrieve class
        llm (object): instance of llm model
        top_k: Number of top results to return
        min_score (float, optional): Minimum similarity score threshold. Defaults to 0.2.
        return_context (bool, optional): flag to return result. Defaults to False.
    """
    results = retriever.retrieve(query, top_k, score_threshold=min_score)
    if not results:
        raise {
            "answer": "No relevent context found.",
            "source": [],
            "confidence": 0.0,
            "context": "",
        }

    # prepare context and sources
    context = "\n\n".join([doc["content"] for doc in results])
    sources = [
        {
            "source": doc["metadata"].get(
                "source_file", doc["metadata"].get("source", "unknown")
            ),
            "page": doc["metadata"].get("page", "unkown"),
            "sorce": doc["similarity_score"],
            "preview": doc["content"][:300] + "...",
        }
        for doc in results
    ]
    confidence = max([doc["similarity_score"] for doc in results])

    # generate answer
    prompt = f"""Use the following context to answer the question concisely.
    \nContext:\n{context}
    \n\nQuestion: {query}
    \n\nAnswer:"""
    response = llm.invoke([prompt.format(context, query)])

    output = {"answer": response.content, "sources": sources, "confidence": confidence}

    if return_context:
        output["context"] = context

    return output


# Run
result = rag_advance(
    "Myopic Nature of Token-level Predictions",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True,
)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence:", result["confidence"])
print("Context Preview:", result["context"][:300])

### Advance RAG Pipeline with Streaming, Citations, History, and Summarization

In [ ]:
from typing import Dict, Any
import time


class AdvanceRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # store query history

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False,
    ) -> Dict[str, any]:
        # retrieve relevent documents
        results = self.retriever.retrieve(question, top_k, score_threshold=min_score)
        if not result:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc["content"] for doc in results])
            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file", doc["metadata"].get("source", "unknown")
                    ),
                    "page": doc["metadata"].get("page", "unknown"),
                    "score": doc["similarity_score"],
                    "preview": doc["content"][:120] + "...",
                }
                for doc in results
            ]

            # streaming answer simulatiokn
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i : i + 80], end="", flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke(
                [prompt.format(context=context, question=question)]
            )
            answer = response.content

            # add ciations to answer
            citations = [
                f"[{i+1}] {src['source']} (page {src['page']})"
                for i, src in enumerate(sources)
            ]
            answer_with_citations = (
                answer + "\n\nCitations:\n" + "\n".join(citations)
                if citations
                else answer
            )

            # optionally summarize answer
            summary = None
            if summarize and answer:
                summary_prompt = (
                    f"Summarize the following answer in 2 sentences:\n{answer}"
                )
                summary_resp = self.llm.invoke([summary_prompt])
                summary = summary_resp.content

            # store query history
            self.history.append(
                {
                    "question": question,
                    "answer": answer,
                    "sources": sources,
                    "summary": summary,
                }
            )

            return {
                "question": question,
                "answer": answer_with_citations,
                "sources": sources,
                "summary": summary,
                "history": self.history,
            }


# run
advance_rag = AdvanceRAGPipeline(rag_retriever, llm)
result = advance_rag.query(
    "What is attention is all you need",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True,
)
print("\nFinal Answer:", result["answer"])
print("Summary:", result["summary"])
print("History:", result["history"][-1])

### RAG Pipeline - VectoDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
class GroqLLM:
    def __init__(self, model_name: str = "qwen/qwen3.8-27b", api_key: str = None):
        """
        Initialize Grop LLM

        Args:
            model_name: Grop model name(qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError(
                "Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter."
            )

        self.llm = ChatGroq(
            api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024,
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context

        Args:
            query (str): user qustion
            context (str): retrieved document context
            max_length (int, optional): maximum response length. Defaults to 500.

        Returns:
            str: generated response string
        """

        # create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

            Context:
            {context}

            Question: {question}

            Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so.""",
        )

        # format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            # generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)

            return response.content
        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple respone generation without complex prompting

        Args:
            query (str): user question
            context (str): retrieved context

        Returns:
            str: generated response
        """
        simple_prompt = (
            f"""Based on this context: {context}\n\nQuestion: {query}\n\n Answer:"""
        )

        try:
            message = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke([simple_prompt])

            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [ ]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

In [ ]:
# get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Training RNNs without recurrence")
